In [0]:

%sql
SHOW CATALOGS;

In [0]:

dbutils.secrets.help()

In [0]:
try:
    api_key = dbutils.secrets.get(catalog="domiciliarycare", schema="security", key="api_key")
except:
    print("Key not found")


In [0]:
# We are using the CQC API.
base_url = "https://api.service.cqc.org.uk/public/v1"
HEADERS = {
    "Ocp-Apim-Subscription-Key": api_key,
    "User-Agent": "HomeSafeKentPipeline/1.0",  
    "Accept": "application/json",
}
print("Key loaded from Unity Catalog secret. Ready for the sanity check below.")

In [0]:
import requests
import time
def cqc_get(path, params=None, max_retries=3):
    """GET against the CQC Syndication API with basic 429 backoff and readable errors."""
    url = f"{base_url}{path}"
    resp = None
    for attempt in range(1, max_retries + 1):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=30)

        if resp.status_code == 200:
            return resp.json()

        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"Rate limited (429). Waiting {wait}s before retry {attempt}/{max_retries}...")
            time.sleep(wait)
            continue

        if resp.status_code == 403:
            raise RuntimeError(
                "403 Forbidden -- confirmed by CQC Developer Support (3 Sept 2026) that this means "
                "the User-Agent header is missing (or, rarely, contains a hyphen). This is NOT an "
                "auth/key problem -- check HEADERS['User-Agent'] is set before touching the key."
            )

        if resp.status_code == 502:
            raise RuntimeError(
                "502 Bad Gateway from CQC's Azure Application Gateway. Verified live on 2 Sept 2026: "
                "this is what a missing/wrong key looked like in that one test, not a CQC outage. "
                "CAVEAT: CQC Developer Support states in writing (3 Sept 2026) that a missing/invalid "
                "key should return 401, not 502 -- if you see 502 with a real key, check "
                "CQC_API_Access_and_Usage_Guide.md section 6 before assuming it's just a bad key."
            )

        if resp.status_code == 401:
            raise RuntimeError(
                "401 Unauthorized -- a key was sent but rejected, per CQC Developer Support this also "
                "covers a missing key. Check it's the Syndication product key specifically (CQC issues "
                "keys per product)."
            )

        resp.raise_for_status()

    raise RuntimeError(f"Gave up after {max_retries} attempts (last status {resp.status_code if resp else 'n/a'}).")

In [0]:
def fetch_location_detail(location_id):
    return cqc_get(f"/locations/{location_id}")

In [0]:
sample = cqc_get("/locations", params={"page": 1, "perPage": 1})
sample

In [0]:
KENT_LOCAL_AUTHORITIES = [
    "Ashford", "Canterbury", "Dartford", "Dover", "Folkestone and Hythe",
    "Gravesham", "Maidstone", "Sevenoaks", "Swale", "Thanet",
    "Tonbridge and Malling", "Tunbridge Wells",
]
# Medway deliberately excluded -- see the decision note above.

print(f"{len(KENT_LOCAL_AUTHORITIES)} Kent districts configured.")

In [0]:
sample_batch = cqc_get("/locations", params={"perPage": 100, "page": 1})

distinct_las = set()
for loc in sample_batch["locations"]:
    detail = fetch_location_detail(loc["locationId"])
    distinct_las.add(detail.get("localAuthority"))
    time.sleep(0.3)

print(sorted(distinct_las))

In [0]:
sample_batch = cqc_get("/locations", params={"perPage": 100, "page": 1})

distinct_activities = set()
for loc in sample_batch["locations"]:
    detail = fetch_location_detail(loc["locationId"])
    activities = detail.get("regulatedActivities", []) or []
    for a in activities:
        distinct_activities.add(a.get("name"))
    time.sleep(0.3)

print(sorted(a for a in distinct_activities if a))

In [0]:

def fetch_all_locations(local_authority="Kent", regulated_activity="Personal care", per_page=1000, polite_delay=0.3):
    """Page through /locations filtered to Kent + Personal care."""
    params = [
        ("localAuthority", local_authority),
        ("regulatedActivity", regulated_activity),
        ("perPage", per_page),
    ]
    all_locations = []
    page = 1
    while True:
        page_params = params + [("page", page)]
        data = cqc_get("/locations", params=page_params)
        all_locations.extend(data["locations"])
        print(f"Page {data['page']}/{data['totalPages']} -- {len(data['locations'])} rows "
              f"(running total {len(all_locations)})")
        if not data.get("nextPageUri"):
            break
        page += 1
        time.sleep(polite_delay)
    return all_locations


kent_locations_summary = fetch_all_locations()
print(f"\nTotal Kent personal-care locations (summary level): {len(kent_locations_summary)}")

In [0]:
def fetch_location_detail(location_id):
    return cqc_get(f"/locations/{location_id}")


detail_records = []
failed_ids = []

for i, loc in enumerate(kent_locations_summary, 1):
    try:
        detail_records.append(fetch_location_detail(loc["locationId"]))
    except Exception as e:
        print(f"FAILED on {loc['locationId']} ({loc.get('locationName')}): {e}")
        failed_ids.append(loc["locationId"])
    if i % 50 == 0:
        print(f"Processed {i}/{len(kent_locations_summary)}...")
    time.sleep(0.3)

print(f"\nDone. Fetched {len(detail_records)} records. Failed: {len(failed_ids)}")
if failed_ids:
    print(f"Failed IDs: {failed_ids}")

In [0]:
def flatten_location_detail(rec):
    ratings = (rec.get("currentRatings") or {}).get("overall") or {}
    overall_rating = ratings.get("rating")
    overall_rating_date = ratings.get("reportDate")
    rating_framework = "legacy_ratings"

    if not overall_rating:
        assessment = rec.get("assessment") or []
        if assessment:
            asg_ratings = assessment[0].get("ratings", {}).get("asgRatings", [])
            if asg_ratings:
                overall_rating = asg_ratings[0].get("rating")
                overall_rating_date = assessment[0].get("assessmentPlanPublishedDateTime")
                rating_framework = "single_assessment"
            else:
                rating_framework = "not_yet_rated"
        else:
            rating_framework = "not_yet_rated"

    use_of_resources = ratings.get("useOfResources", {}) or {}
    service_ratings = (rec.get("currentRatings") or {}).get("serviceRatings", []) or []
    inspection_areas = rec.get("inspectionAreas", []) or []
    superseded_areas = [ia.get("inspectionAreaName") for ia in inspection_areas if ia.get("status") == "Superseded"]
    active_areas = [ia.get("inspectionAreaName") for ia in inspection_areas if ia.get("status") not in ("Superseded", None)]

    return {
        "location_id": rec.get("locationId"),
        "provider_id": rec.get("providerId"),
        "location_name": rec.get("name"),
        "postal_code": rec.get("postalCode"),
        "town_city": rec.get("postalAddressTownCity"),
        "local_authority": rec.get("localAuthority"),
        "region": rec.get("region"),
        "registration_status": rec.get("registrationStatus"),
        "registration_date": rec.get("registrationDate"),
        "regulated_activities": "; ".join(a.get("name", "") for a in rec.get("regulatedActivities", []) or []),
        "gac_service_types": "; ".join(s.get("name", "") for s in rec.get("gacServiceTypes", []) or []),
        "overall_rating": overall_rating,
        "overall_rating_date": overall_rating_date,
        "rating_framework": rating_framework,
        "number_of_beds": rec.get("numberOfBeds"),
        "use_of_resources_rating": use_of_resources.get("useOfResourcesRating"),
        "combined_quality_rating": use_of_resources.get("combinedQualityRating"),
        "service_ratings": "; ".join(f"{s.get('name')}: {s.get('rating')}" for s in service_ratings),
        "inspection_areas_active": "; ".join(a for a in active_areas if a),
        "inspection_areas_superseded": "; ".join(a for a in superseded_areas if a),
    }


In [0]:
import pandas as pd
df = pd.DataFrame([flatten_location_detail(r) for r in detail_records])
print(f"Flattened {len(df)} rows, {len(df.columns)} columns")
df.head()

In [0]:
for col in ["number_of_beds", "use_of_resources_rating", "combined_quality_rating", "service_ratings"]:
    non_blank = (df[col].astype(str).str.strip() != "").sum() & (df[col].notna()).sum()
    print(f"{col}: non-blank count = {(df[col].astype(str).str.strip().replace('None','')  != '').sum()} / {len(df)}")

In [0]:
spark_df = spark.createDataFrame(df.astype(str))
spark_df.write.format("delta").mode("overwrite") \
    .saveAsTable("domiciliarycare.bronze.cqc_kent_locations")

print(f"Saved {len(df)} rows to domiciliarycare.bronze.cqc_kent_locations")

In [0]:

def fetch_provider_detail(provider_id):
    return cqc_get(f"/providers/{provider_id}")


def flatten_provider_detail(rec):
    return {
        "provider_id": rec.get("providerId"),
        "provider_name": rec.get("name"),
        "ownership_type": rec.get("ownershipType"),
        "companies_house_number": rec.get("companiesHouseNumber"),
        "charity_number": rec.get("charityNumber"),
        "registration_status": rec.get("registrationStatus"),
        "registration_date": rec.get("registrationDate"),
        "postal_code": rec.get("postalCode"),
        "local_authority": rec.get("localAuthority"),
        "region": rec.get("region"),
    }


# Get every distinct provider ID behind your 457 Kent locations
kent_provider_ids = sorted({r.get("providerId") for r in detail_records if r.get("providerId")})
print(f"{len(kent_provider_ids)} distinct providers behind {len(detail_records)} Kent locations")

provider_records = []
failed_provider_ids = []

for i, pid in enumerate(kent_provider_ids, 1):
    try:
        provider_records.append(fetch_provider_detail(pid))
    except Exception as e:
        print(f"FAILED on provider {pid}: {e}")
        failed_provider_ids.append(pid)
    if i % 50 == 0:
        print(f"Fetched {i}/{len(kent_provider_ids)} providers...")
    time.sleep(0.3)

providers_df = pd.DataFrame([flatten_provider_detail(r) for r in provider_records])
print(f"\nFetched {len(providers_df)} provider records")

# This is the key Week 2 number -- check it before deciding on fuzzy matching
fill_rate = providers_df["companies_house_number"].notna().mean()
print(f"companies_house_number fill rate: {fill_rate:.1%}")
providers_df.head()

In [0]:
spark_providers = spark.createDataFrame(providers_df.astype(str))
spark_providers.write.format("delta").mode("overwrite") \
    .saveAsTable("domiciliarycare.bronze.cqc_kent_providers")

print(f"Saved {len(providers_df)} provider rows to domiciliarycare.bronze.cqc_kent_providers")